# Demo – Chatbot pháp luật (RAG)
Cần GPU để chạy LLM. Nếu không có GPU, đặt `LLM = "extractive"` để xem phần retrieval.

In [1]:
# Settings -> Accelerator: GPU (T4 x2 or P100), Internet: ON
import os, subprocess
REPO = "https://github.com/dinhthithuthuyduphong-bot/LAB.git"
BRANCH = "RAG-based-chatbot-system"
WORK = "/kaggle/working/LAB" if os.path.exists("/kaggle") else "/content/LAB"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)
os.chdir(WORK)
print("working dir:", os.getcwd())
# Large artifacts (chunks / BM25 / FAISS) can be kept in a Kaggle Dataset or Google Drive between sessions:
# os.environ["ARTIFACTS_DIR"] = "/kaggle/input/lab-rag-artifacts"   # read-only input -> copy if you need to write
!pip install -q -r requirements-gpu.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

Cloning into '/kaggle/working/LAB'...


working dir: /kaggle/working/LAB


Updating files: 100% (16552/16552), done.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import os
for _d in ("/kaggle/working/LAB", "/content/LAB"):
    if os.path.isdir(_d):
        os.chdir(_d)
        break
else:
    raise SystemExit("Repo not found: run the first (setup) cell once")
import os
if not os.path.exists("artifacts/bm25_legal"):
    !python -I scripts/build_corpus.py && python -I scripts/prepare_data.py && python -I scripts/build_chunks.py --strategy legal && python -I scripts/build_bm25.py --chunks legal

Cleaning corpus: 100%|███████████████████| 16439/16439 [01:14<00:00, 221.71it/s]
16439 documents -> /kaggle/working/LAB/artifacts/docs.jsonl
total chars: raw=329,096,133 cleaned=268,249,709
res.csv: 2745 rows | cited doc in corpus: 51 rows, 8/315 unique docs
eval_set.csv: 200 rows ({'not_covered': 149, 'covered': 51}), seed=42
fewshot_pool.csv: 2545 rows
Chunking (legal, 1000/150): 100%|████████| 16439/16439 [00:29<00:00, 565.15it/s]
384,612 chunks -> /kaggle/working/LAB/artifacts/chunks_legal.jsonl
chunk length: mean=787 median=882 p95=997 max=1172
BM25 tokenising: 100%|████████████████| 384612/384612 [02:43<00:00, 2349.93it/s]
BM25 index over 384,612 chunks -> /kaggle/working/LAB/artifacts/bm25_legal  (183s)


In [3]:
import os
for _d in ("/kaggle/working/LAB", "/content/LAB"):
    if os.path.isdir(_d):
        os.chdir(_d)
        break
else:
    raise SystemExit("Repo not found: run the first (setup) cell once")
import sys; sys.path.insert(0, "."); sys.path.insert(0, "app"); sys.path.insert(0, "scripts")
from demo import Bot, EXAMPLES
LLM = "Qwen/Qwen2.5-3B-Instruct"     # or "extractive"
RETRIEVER = "bm25_legal"             # "hybrid_legal_rerank" once the FAISS index is built
bot = Bot(RETRIEVER, LLM)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
import os
for _d in ("/kaggle/working/LAB", "/content/LAB"):
    if os.path.isdir(_d):
        os.chdir(_d)
        break
else:
    raise SystemExit("Repo not found: run the first (setup) cell once")
from IPython.display import Markdown, display
for q in EXAMPLES:
    ans, src = bot.answer(q)
    display(Markdown(f"### ❓ {q}\n\n{ans}\n\n<details><summary>Nguồn</summary>\n\n{src}\n</details>"))

### ❓ Mức lương cơ sở áp dụng từ ngày 01/7/2024 đối với cán bộ, công chức, viên chức là bao nhiêu?

Căn cứ theo Điều 3, Mục 2 của Nghị định 73/2024/NĐ-CP quy định mức lương cơ sở và chế độ tiền thưởng đối với cán bộ, công chức, viên chức và lực lượng vũ trang, mức lương cơ sở áp dụng từ ngày 01/7/2024 là 2.340.000 đồng/tháng.

<details><summary>Nguồn</summary>

**[1]** [Kết luận 83-KL/TW về cải cách tiền lương; điều chỉnh lương hưu, trợ cấp bảo hiểm xã hội, trợ cấp ưu đãi người có công và trợ cấp xã hội từ ngày 01/7/2024 do Ban Chấp hành Trung ương ban hành] — `83-KL_TW.txt`
> 2. Về nội dung cải cách tiền lương theo Nghị quyết số 27-NQ/TW
2.1. Đối với khu vực doanh nghiệp
Thực hiện đầy đủ 2 nội dung: (1) Điều chỉnh mức lương tối thiểu vùng (theo tháng và theo giờ) theo quy định của Bộ luật Lao động (tăng 6% so với năm 2023), áp dụng từ ngày 01/7/2024. (2) Quy định cơ chế …
**[2]** [Kết luận 83-KL/TW về cải cách tiền lương; điều chỉnh lương hưu, trợ cấp bảo hiểm xã hội, trợ cấp ưu đãi người có công và trợ cấp xã hội từ ngày 01/7/2024 do Ban Chấp hành Trung ương ban hành] — `83-KL_TW.txt`
> (1) Thực hiện điều chỉnh mức lương cơ sở từ 1,8 triệu đồng lên 2,34 triệu đồng (tăng 30%) kể từ ngày 01/7/2024.
(2) Thực hiện chế độ tiền thưởng từ ngày 01/7/2024 (quỹ tiền thưởng bằng 10% quỹ lương cơ bản). Quy định chế độ tiền thưởng gắn với trách nhiệm người đứng đầu cơ quan, đơn vị trong việc đá…
**[3]** [Quyết định 1150/QĐ-UBND năm 2014 xếp mức phụ cấp đặc thù đối với cán bộ, công chức làm việc tại Ban Bảo vệ, chăm sóc sức khỏe cán bộ tỉnh Bình Phước] — `1150_QĐ-UBND_3.txt`
> Điều 1.
1. Cán bộ, công chức, viên chức làm việc tại Ban Bảo vệ, chăm sóc sức khỏe cán bộ tỉnh được hưởng mức phụ cấp đặc thù như sau:
a) Mức phụ cấp 2,0 lần mức lương cơ sở áp dụng đối với cán bộ, công chức, viên chức có học hàm, học vị Tiến sĩ, Thạc sĩ, bác sĩ Chuyên khoa I, bác sĩ Chuyên khoa II.…
**[4]** [Kết luận 83-KL/TW về cải cách tiền lương; điều chỉnh lương hưu, trợ cấp bảo hiểm xã hội, trợ cấp ưu đãi người có công và trợ cấp xã hội từ ngày 01/7/2024 do Ban Chấp hành Trung ương ban hành] — `83-KL_TW.txt`
> (6) Thực hiện tiền lương và thu nhập của các cơ quan, đơn vị đang được áp dụng cơ chế tài chính và thu nhập đặc thù:
Giao Ban cán sự đảng Chính phủ: Chỉ đạo Chính phủ, các bộ, các cơ quan liên quan tiếp tục rà soát toàn bộ khung khổ pháp lý để trình cấp có thẩm quyền xem xét, quyết định việc sửa đổi…
**[5]** [Nghị định 73/2024/NĐ-CP quy định mức lương cơ sở và chế độ tiền thưởng đối với cán bộ, công chức, viên chức và lực lượng vũ trang] | Điều 3. Mức lương cơ sở — `73_2024_NĐ-CP.txt`
> c) Tính các khoản trích và các chế độ được hưởng theo mức lương cơ sở.
2. Từ ngày 01 tháng 7 năm 2024, mức lương cơ sở là 2.340.000 đồng/tháng.
3. Đối với các cơ quan, đơn vị đang được áp dụng các cơ chế tài chính, thu nhập đặc thù ở trung ương: Thực hiện bảo lưu phần chênh lệch giữa tiền lương và t…
</details>

### ❓ Năm 2025 công chức, viên chức được nghỉ Tết Âm lịch bao nhiêu ngày, từ ngày nào đến ngày nào?

Căn cứ tại Điều 1 Thông báo 1202/TB-UBND năm 2024 quy định:

1. Cán bộ, công chức, viên chức và người lao động của các cơ quan hành chính, sự nghiệp, tổ chức chính trị, tổ chức chính trị - xã hội (sau đây gọi là công chức, viên chức) nghỉ Tết Âm lịch năm 2025 liền 09 ngày liên tục, từ thứ Bảy ngày 25/01/2025 đến hết Chủ nhật ngày 02/02/2025 (tức từ ngày 26 tháng Chạp năm Giáp Thìn đến hết ngày mùng 5 tháng Giêng năm Ất Tỵ).

Vậy, công chức, viên chức được nghỉ Tết Âm lịch năm 2025 là 09 ngày, từ ngày 25/01/2025 đến hết ngày 02/02/2025.

<details><summary>Nguồn</summary>

**[1]** [Thông báo 1202/TB-UBND năm 2024 về nghỉ Tết Âm lịch, nghỉ lễ Quốc khánh, nghỉ lễ ngày Chiến thắng 30/4 và ngày Quốc tế lao động 01/5 năm 2025 do Thành phố Hà Nội ban hành] — `1202_TB-UBND.txt`
> tổ chức chính trị, tổ chức chính trị - xã hội (sau đây gọi là công chức, viên chức) nghỉ Tết Âm lịch và một số dịp nghỉ lễ trong năm 2025, cụ thể:
a) Nghỉ Tết Âm lịch năm 2025 liền 09 ngày liên tục, từ thứ Bảy ngày 25/01/2025 đến hết Chủ nhật ngày 02/02/2025 (tức từ ngày 26 tháng Chạp năm Giáp Thìn …
**[2]** [Thông báo 6150/TB-BLĐTBXH năm 2024 nghỉ tết Âm lịch, nghỉ lễ Quốc khánh, nghỉ lễ ngày Chiến thắng 30/4 và ngày Quốc tế lao động 01/5 năm 2025 đối với cán bộ, công chức, viên chức và người lao động do Bộ Lao động - Thương] — `6150_TB-BLĐTBXH.txt`
> dụng lao động khác thực hiện lịch nghỉ tết Âm lịch, nghỉ lễ Quốc khánh, nghỉ lễ ngày Chiến thắng 30/4 và ngày Quốc tế lao động 01/5 năm 2025 như sau:
1. Cán bộ, công chức, viên chức và người lao động của các cơ quan hành chính, sự nghiệp, tổ chức chính trị, tổ chức chính trị - xã hội (sau đây gọi tắ…
**[3]** [Thông báo 4724/TB-LĐTBXH năm 2014 về hoán đổi ngày nghỉ, ngày đi làm bù các dịp nghỉ lễ, Tết năm 2015 do Bộ Lao động - Thương binh và Xã hội ban hành] — `4724_TB-LĐTBXH.txt`
> nghỉ thứ Sáu, ngày 02/01/2015. Như vậy, dịp nghỉ Tết Dương lịch, công chức, viên chức nghỉ liền 04 ngày, từ ngày 01/01/2015 đến hết ngày 04/01/2015.
2. Dịp nghỉ Tết Âm lịch
Công chức, viên chức nghỉ Tết Âm lịch từ thứ Ba, ngày 17/02/2015 đến hết thứ Bảy, ngày 21/02/2015 (tức là từ ngày 29 tháng chạp…
**[4]** [Thông báo 4699/TB-LĐTBXH năm 2016 về nghỉ Tết Âm lịch năm 2017 đối với cán bộ, công, viên chức và người lao động của cơ quan hành chính, sự nghiệp, tổ chức chính trị, tổ chức chính trị - xã hội do Bộ Lao động - Thương bi] — `4699_TB-LĐTBXH.txt`
> hội thông báo để các cơ quan hành chính, sự nghiệp, tổ chức chính trị, tổ chức chính trị - xã hội thực hiện lịch nghỉ Tết Âm lịch năm 2017 như sau:
1. Cán bộ, công chức, viên chức và người lao động của các cơ quan hành chính, sự nghiệp, tổ chức chính trị, tổ chức chính trị - xã hội (sau đây gọi là c…
**[5]** [Thông báo 1202/TB-UBND năm 2024 về nghỉ Tết Âm lịch, nghỉ lễ Quốc khánh, nghỉ lễ ngày Chiến thắng 30/4 và ngày Quốc tế lao động 01/5 năm 2025 do Thành phố Hà Nội ban hành] — `1202_TB-UBND.txt`
> ỦY BAN NHÂN DÂN
THÀNH PHỐ HÀ NỘI
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
Số: 1202/TB-UBND
Hà Nội, ngày 05 tháng 12 năm 2024
THÔNG BÁO
VỀ VIỆC NGHỈ TẾT ÂM LỊCH, NGHỈ LỄ QUỐC KHÁNH, NGHỈ LỄ NGÀY CHIẾN THẮNG 30/4 VÀ NGÀY QUỐC TẾ LAO ĐỘNG 01/5 NĂM 2025
Thực hiện Thông báo số 6150/…
</details>

### ❓ Người lao động được nghỉ lễ, tết những ngày nào theo Bộ luật Lao động?

Căn cứ theo Điều 115 của Bộ luật Lao động 2019 quy định như sau:

1. Người lao động được nghỉ làm việc, hưởng nguyên lương trong những ngày nghỉ lễ, tết theo quy định tại Điều 115 của Bộ luật Lao động.

2. Theo Điều 115 của Bộ luật Lao động 2019, người lao động được nghỉ làm việc, hưởng nguyên lương trong những ngày nghỉ lễ, tết quy định tại Điều 115 của Bộ luật Lao động.

Như vậy, người lao động được nghỉ làm việc, hưởng nguyên lương trong những ngày nghỉ lễ, tết theo quy định tại Điều 115 của Bộ luật Lao động 2019.

<details><summary>Nguồn</summary>

**[1]** [Quyết định 1523/QĐ-UBND năm 2019 về Quy chế quy định thời giờ làm việc, thời giờ nghỉ ngơi đối với người lao động làm công việc có tính chất đặc biệt trong lĩnh vực phòng chống thiên tai tỉnh Lào Cai] — `1523_QĐ-UBND_1.txt`
> Điều 11. Nghỉ hàng tuần, nghỉ lễ, tết
Nghỉ hàng tuần, nghỉ lễ, tết quy định tại Điều 8 Thông tư số 31/2017/TT-BNNPTNT ngày 29/12/2017 của Bộ Nông nghiệp và PTNT Quy định thời giờ làm việc, thời giờ nghỉ ngơi đối với người lao động làm công việc có tính chất đặc biệt trong lĩnh vực phòng chống thiên …
**[2]** [Thông tư 31/2017/TT-BNNPTNT quy định về thời giờ làm việc, thời giờ nghỉ ngơi đối với người lao động làm công việc có tính chất đặc biệt trong lĩnh vực phòng chống thiên tai do Bộ trưởng Bộ Nông nghiệp và Phát triển nông] — `31_2017_TT-BNNPTNT.txt`
> Điều 8. Nghỉ hàng tuần, nghỉ lễ, tết
1. Người lao động làm công việc có tính chất đặc biệt trong lĩnh vực phòng chống thiên tai được nghỉ hàng tuần, nghỉ lễ, tết và những ngày nghỉ được hưởng lương khác theo quy định của Bộ Luật lao động và các văn bản hướng dẫn thi hành Bộ Luật Lao động.
2. Trường …
**[3]** [Thông tư 23/2015/TT-BLĐTBXH hướng dẫn thực hiện về tiền lương của Nghị định 05/2015/NĐ-CP quy định chi tiết và hướng dẫn thi hành một số nội dung của Bộ luật Lao động do Bộ trưởng Bộ Lao động - Thương binh và Xã hội ban ] | Điều 6. Tiền lương làm thêm giờ — `23_2015_TT-BLĐTBXH.txt`
> b) Mức ít nhất bằng 150% so với tiền lương giờ thực trả của ngày làm việc bình thường, áp dụng đối với giờ làm thêm vào ngày thường;
c) Mức ít nhất bằng 200% so với tiền lương giờ thực trả của ngày làm việc bình thường, áp dụng đối với giờ làm thêm vào ngày nghỉ hằng tuần;
d) Mức ít nhất bằng 300% s…
**[4]** [Nghị định 27/2014/NĐ-CP hướng dẫn Bộ luật Lao động về lao động là người giúp việc gia đình] — `27_2014_NĐ-CP.txt`
> Điều 23. Nghỉ hằng năm, nghỉ lễ, tết
1. Người lao động có đủ 12 tháng làm việc cho một người sử dụng lao động thì được nghỉ hằng năm là 12 ngày làm việc và được hưởng nguyên lương. Thời điểm nghỉ do hai bên thỏa thuận. Người lao động có thể thỏa thuận với người sử dụng lao động để nghỉ hằng năm thàn…
**[5]** [Thông tư 21/2015/TT-BGTVT Quy định về thời giờ làm việc và thời giờ nghỉ ngơi đối với người lao động làm công việc có tính chất đặc biệt trong vận tải đường sắt do Bộ trưởng Bộ Giao thông vận tải ban hành] — `21_2015_TT-BGTVT.txt`
> Điều 6. Quy định về nghỉ hàng tuần, hàng năm, nghỉ lễ tết và nghỉ khác
Các chức danh làm việc theo ban và làm việc trên đoàn tàu quy định tại Điều 4 và Điều 5 của Thông tư này được nghỉ hàng tuần, nghỉ hàng năm, nghỉ lễ tết và những ngày nghỉ được hưởng lương khác theo quy định của Bộ luật Lao động …
</details>

### ❓ Chế độ tiền thưởng theo Nghị định 73/2024/NĐ-CP được thực hiện như thế nào?

Căn cứ tại Điều 4 của Nghị định 73/2024/NĐ-CP quy định mức lương cơ sở và chế độ tiền thưởng đối với cán bộ, công chức, viên chức và lực lượng vũ trang, cụ thể như sau:

Chế độ tiền thưởng theo Nghị định 73/2024/NĐ-CP không bao gồm phụ cấp. Tiền thưởng này được áp dụng cho chức vụ, chức danh, ngạch, bậc và cấp bậc quân hàm của các đối tượng trong danh sách trả lương của cơ quan, đơn vị. 

Căn cứ tại Điều 5 của Nghị định 73/2024/NĐ-CP, Chính phủ, cơ quan khác ở trung ương và các tỉnh, thành phố trực thuộc trung ương sau khi đã thực hiện các quy định tại khoản 1 và khoản 2 Điều này. Điều này cho thấy rằng chế độ tiền thưởng được thực hiện sau khi các quy định về mức lương cơ sở và chế độ tiền thưởng đã được thực hiện.

Đến hết ngày 31 tháng 01 của năm sau liền kề, nếu cơ quan, đơn vị không sử dụng hết quỹ tiền thưởng của năm thì không được chuyển nguồn vào quỹ tiền thưởng của năm sau. Điều này nhằm đảm bảo việc sử dụng hiệu quả và tiết kiệm ngân sách nhà nước.

Vậy chế độ tiền thưởng theo Nghị định 73/2024/NĐ-CP được thực hiện dựa trên chức vụ, chức danh, ngạch, bậc và cấp bậc quân hàm của các đối tượng trong danh sách trả lương của cơ quan, đơn vị, và được thực hiện sau khi các quy định về mức lương cơ sở và chế độ tiền thưởng đã được thực hiện.

<details><summary>Nguồn</summary>

**[1]** [Nghị định 73/2024/NĐ-CP quy định mức lương cơ sở và chế độ tiền thưởng đối với cán bộ, công chức, viên chức và lực lượng vũ trang] | Điều 4. Chế độ tiền thưởng — `73_2024_NĐ-CP.txt`
> (không bao gồm phụ cấp) theo chức vụ, chức danh, ngạch, bậc và cấp bậc quân hàm của các đối tượng trong danh sách trả lương của cơ quan, đơn vị.
Đến hết ngày 31 tháng 01 của năm sau liền kề, nếu cơ quan, đơn vị không sử dụng hết quỹ tiền thưởng của năm thì không được chuyển nguồn vào quỹ tiền thưởng…
**[2]** [Nghị định 73/2024/NĐ-CP quy định mức lương cơ sở và chế độ tiền thưởng đối với cán bộ, công chức, viên chức và lực lượng vũ trang] — `73_2024_NĐ-CP.txt`
> CHÍNH PHỦ
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
Số: 73/2024/NĐ-CP
Hà Nội, ngày 30 tháng 6 năm 2024
NGHỊ ĐỊNH
QUY ĐỊNH MỨC LƯƠNG CƠ SỞ VÀ CHẾ ĐỘ TIỀN THƯỞNG ĐỐI VỚI CÁN BỘ, CÔNG CHỨC, VIÊN CHỨC VÀ LỰC LƯỢNG VŨ TRANG
Căn cứ Luật Tổ chức Chính phủ ngày 19 tháng 6 năm 2015; Luật…
**[3]** [Nghị định 73/2024/NĐ-CP quy định mức lương cơ sở và chế độ tiền thưởng đối với cán bộ, công chức, viên chức và lực lượng vũ trang] | Điều 5. Kinh phí thực hiện — `73_2024_NĐ-CP.txt`
> Chính phủ, cơ quan khác ở trung ương và các tỉnh, thành phố trực thuộc trung ương sau khi đã thực hiện các quy định tại khoản 1 và khoản 2 Điều này.
4. Kinh phí thực hiện cải cách tiền lương, thực hiện chế độ tiền thưởng của viên chức, người lao động trong các đơn vị sự nghiệp công lập nhóm 1, nhóm …
</details>

### ❓ Điều kiện cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam là gì?

Theo các nguồn văn bản pháp luật được cung cấp, chưa có quy định cụ thể về các điều kiện cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam. Các văn bản chỉ đề cập đến thủ tục hành chính liên quan đến việc cấp, cấp lại, gia hạn giấy phép lao động, lệ phí cấp giấy phép lao động, và trách nhiệm của người sử dụng lao động trong việc thực hiện các quy định lao động và nộp lệ phí.

Vì vậy, không thể trả lời trực tiếp về điều kiện cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam dựa trên các văn bản này. Để có thông tin chính xác về điều kiện cấp giấy phép lao động, cần tham khảo các văn bản pháp luật liên quan đến lĩnh vực lao động và quản lý người nước ngoài làm việc tại Việt Nam hiện hành.

<details><summary>Nguồn</summary>

**[1]** [Nghị quyết 32/2022/NQ-HĐND quy định mức thu, kê khai, nộp và quản lý lệ phí cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam áp dụng trên địa bàn tỉnh Gia Lai] — `32_2022_NQ-HĐND.txt`
> Điều 2. Mức thu lệ phí cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam
a) Lệ phí cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam: 600.000 đồng/giấy phép.
b) Lệ phí cấp lại giấy phép lao động cho người nước ngoài làm việc tại Việt Nam: 450.000 đồng/giấy phép.
c) Lệ…
**[2]** [Quyết định 3338/QĐ-UBND năm 2016 công bố thủ tục hành chính thuộc thẩm quyền giải quyết của Sở Lao động - Thương binh và Xã hội tỉnh Khánh Hòa] | Điều 4. Chánh Văn phòng Ủy ban nhân dân tỉnh, Giám đốc Sở Lao động-Thương binh và Xã hội, Thủ trưởng các sở, ban, ngành; Chủ tịch UBND các huyện, thị xã, thành  — `3338_QĐ-UBND.txt`
> IV. LĨNH VỰC VIỆC LÀM (14)
62
Cấp mới Giấy phép lao động cho người nước ngoài làm việc tại Việt Nam
63
Cấp lại Giấy phép lao động cho người nước ngoài làm việc tại Việt Nam trường hợp giấy phép lao động bị mất
64
Cấp lại Giấy phép lao động cho người nước ngoài làm việc tại Việt Nam trường hợp giấy p…
**[3]** [Nghị định 58-CP năm 1996 về cấp giấy phép lao động cho người nước ngoài làm việc tại các doanh nghiệp, tổ chức ở Việt Nam] — `58-CP_1.txt`
> Chương 2:
ĐIỀU KIỆN VÀ THỦ TỤC TUYỂN, CẤP GIẤY PHÉP CHO NGƯỜI NƯỚC NGOÀI LÀM VIỆC TẠI VIỆT NAM…
**[4]** [Quyết định 1939/QĐ-UBND năm 2009 về công bố bộ thủ tục hành chính thuộc thẩm quyền giải quyết của Sở Lao động - Thương binh và Xã hội tỉnh Khánh Hòa do Ủy ban nhân dân tỉnh Khánh Hòa ban hành] | Điều 4. Chánh Văn phòng Ủy ban nhân dân tỉnh, Giám đốc Sở Lao động - Thương binh và Xã hội, Tổ trưởng Tổ công tác thực hiện Đề án 30 của tỉnh và thủ trưởng các  — `1939_QĐ-UBND_1.txt`
> 2
Duyệt đơn giá tiền lương doanh nghiệp nhà nước
III. Lĩnh vực việc làm
Cấp sổ lao động
Cấp giấy phép lao động cho người nước ngoài làm việc tại Việt Nam
Cấp giấy phép lao động cho người nước ngoài di chuyển nội bộ doanh nghiệp
Cấp giấy phép lao động cho người nước ngoài thực hiện các loại hợp đồng …
**[5]** [Nghị định 34/2008/NĐ-CP quy định về tuyển dụng và quản lý người nước ngoài làm việc tại Việt Nam] — `34_2008_NĐ-CP.txt`
> Điều 19. Trách nhiệm của người sử dụng lao động, đối tác phía Việt Nam và tổ chức phi chính phủ nước ngoài
1. Thực hiện đúng các quy định của pháp luật lao động Việt Nam và các quy định pháp luật khác có liên quan của Việt Nam.
2. Làm các thủ tục đề nghị cấp giấy phép lao động, gia hạn giấy phép lao…
</details>

In [3]:
import os, sys, subprocess
REPO, BRANCH = "https://github.com/dinhthithuthuyduphong-bot/LAB.git", "RAG-based-chatbot-system"
WORK = "/kaggle/working/LAB" if os.path.exists("/kaggle") else "/content/LAB"
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "pull", "-q"], check=False)   # lấy bản demo mới nhất
os.chdir(WORK)
!pip install -q -r requirements-gpu.txt

import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "→ CHƯA BẬT GPU")

if not os.path.exists("artifacts/bm25_legal"):          # lần đầu: khoảng 10 phút
    !python -I scripts/build_corpus.py && python -I scripts/prepare_data.py && python -I scripts/build_chunks.py --strategy legal && python -I scripts/build_bm25.py --chunks legal

sys.path[:0] = [".", "app", "scripts"]
from demo import Bot
bot = Bot("bm25_legal", "Qwen/Qwen2.5-3B-Instruct")    # tải model: khoảng 2–3 phút

from notebook_ui import notebook_ui
notebook_ui(bot)

Cloning into '/kaggle/working/LAB'...
Updating files: 100% (16553/16553), done.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
GPU: True Tesla T4
Cleaning corpus: 100%|███████████████████| 16439/16439 [01:00<00:00, 272.05it/s]
16439 documents -> /kaggle/working/LAB/artifacts/docs.jsonl
total chars: raw=329,096,133 cleaned=268,249,709
res.csv: 2745 rows | cited doc in corpus: 51 rows, 8/315 unique docs
eval_set.csv: 200 rows ({'not_covered': 149, 'covered': 51}), seed=42
fewshot_pool.csv: 2545 rows
Chunking (legal, 1000/150): 100%|████████| 16439/16439 [00:22<00:00, 731.59it/s]
384,612 chunks -> /kaggle/working/LAB/artifacts/chunks_legal.jsonl
chunk length: mean=787 median=882 p95=997 max=1172
BM25 tokenising: 100%|████████████████| 384612/384612 [02:12<00:00, 2902.61it/s]
BM25 index over 384,612 chunks -> /kaggle/working/LAB/artifacts/bm25_legal  (147s)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM Qwen/Qwen2.5-3B-Instruct loaded on: cuda:0
